# Watermarking revision experiments (Google Colab)
Run the cells from top to bottom. Results are saved to Google Drive (`MyDrive/wm_revision`), so nothing is lost if Colab disconnects.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/wm_revision'
os.makedirs(OUT, exist_ok=True)
%cd /content

## 2. Upload the two scripts
Select **wm_classical.py** and **wm_learned.py** in the dialog. A copy is also stored in Drive; if you rerun the notebook later, the cell reuses that copy without asking again.

In [ ]:
import shutil
needed = ['wm_classical.py', 'wm_learned.py']
if all(os.path.exists(f'{OUT}/{f}') for f in needed):
    for f in needed: shutil.copy(f'{OUT}/{f}', f)
    print('Scripts copied from Drive')
else:
    from google.colab import files
    files.upload()
    for f in needed: shutil.copy(f, f'{OUT}/{f}')
!ls -l wm_*.py

## 3. Download MS COCO val2017 (about 1 GB, 5000 images)

In [ ]:
if not os.path.isdir('val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip
    !unzip -q val2017.zip && rm val2017.zip
!ls val2017 | wc -l

## 4. Install libraries

In [ ]:
!pip install -q numpy scipy opencv-python-headless scikit-image pandas scikit-learn joblib
import platform, sklearn, cv2, numpy
print(platform.python_version(), 'sklearn', sklearn.__version__, 'opencv', cv2.__version__, 'numpy', numpy.__version__)

## 5. Quick test (about 5 minutes)
If a table appears at the end, everything works. If there is an error, send the error text.

In [ ]:
!python wm_classical.py --images val2017 --n 50 --out sinaq

## 6. Classical methods (DCT-DWT and DCT-DWT-SVD)

In [ ]:
!python wm_classical.py --images val2017 --n 1000 --out "$OUT/results"

## 7. Neural-network extraction (about 1-1.5 hours)
Keep this browser tab open while it runs.

In [ ]:
!python wm_learned.py --images val2017 --n 1000 --out "$OUT/results_nn"

## 8. Download the files to send

In [ ]:
import zipfile
send = ['results/summary_table.md', 'results/config.json', 'results/image_ids.txt',
        'results_nn/summary_table.md', 'results_nn/config.json', 'results_nn/training_log.csv']
with zipfile.ZipFile('send_results.zip', 'w') as z:
    for f in send: z.write(f'{OUT}/{f}', f)
from google.colab import files
files.download('send_results.zip')